# Étape 2 : comprendre le jeu Olist

Objectif : ouvrir les 9 fichiers, comprendre **comment ils sont liés**, **ce que contient chaque colonne d'important**
et **comment le garder par agrégation**. Les conclusions sont reportées dans `Carnet.md`.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Le notebook peut être lancé depuis la racine du dépôt ou depuis notebooks/
DATA = next(p for p in [Path("data"), Path("../data")] if p.exists())
DATA.resolve()

PosixPath('/home/loris/Documents/School/TraitementFlux/OLIST/data')

## 1. Chargement des 9 fichiers

In [2]:
fichiers = {
    "customers":   "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv",
}
# Les codes postaux sont lus en texte : ce sont des identifiants, pas des nombres
# (le zéro initial de '01037' serait perdu en int).
zip_cols = {c: str for c in ["customer_zip_code_prefix", "seller_zip_code_prefix", "geolocation_zip_code_prefix"]}
dfs = {nom: pd.read_csv(DATA / f, dtype=zip_cols) for nom, f in fichiers.items()}

customers, geolocation, orders, order_items, payments, reviews, products, sellers, translation = dfs.values()

pd.DataFrame({
    "lignes":   {n: d.shape[0] for n, d in dfs.items()},
    "colonnes": {n: d.shape[1] for n, d in dfs.items()},
    "Mo":       {n: round(d.memory_usage(deep=True).sum() / 1e6, 1) for n, d in dfs.items()},
})

,lignes,colonnes,Mo
customers,99441,5,36.4
geolocation,1000163,5,206.3
orders,99441,8,61.8
order_items,112650,7,41.3
payments,103886,5,18.7
reviews,99224,7,44.8
products,32951,9,7.1
sellers,3095,4,0.9
translation,71,2,0.0


## 2. Vue d'ensemble de chaque table (types, manquants, cardinalité)

In [3]:
def profil(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "manquants": df.isna().sum(),
        "% manquants": (df.isna().mean() * 100).round(1),
        "valeurs distinctes": df.nunique(),
        "exemple": df.iloc[0],
    })

for nom, df in dfs.items():
    print(f"\n===== {nom}  {df.shape}  doublons complets : {df.duplicated().sum()}")
    display(profil(df))


===== customers  (99441, 5)  doublons complets : 0


,dtype,manquants,% manquants,valeurs distinctes,exemple
customer_id,str,0,0.0,99441,06b8999e2fba1a1fbc88172c00ba8bc7
customer_unique_id,str,0,0.0,96096,861eff4711a542e4b93843c6dd7febb0
customer_zip_code_prefix,str,0,0.0,14994,14409
customer_city,str,0,0.0,4119,franca
customer_state,str,0,0.0,27,SP



===== geolocation  (1000163, 5)  doublons complets : 261831


,dtype,manquants,% manquants,valeurs distinctes,exemple
geolocation_zip_code_prefix,str,0,0.0,19015,01037
geolocation_lat,float64,0,0.0,717372,-23.545621
geolocation_lng,float64,0,0.0,717615,-46.639292
geolocation_city,str,0,0.0,8011,sao paulo
geolocation_state,str,0,0.0,27,SP



===== orders  (99441, 8)  doublons complets : 0


,dtype,manquants,% manquants,valeurs distinctes,exemple
order_id,str,0,0.0,99441,e481f51cbdc54678b7cc49136f2d6af7
customer_id,str,0,0.0,99441,9ef432eb6251297304e76186b10a928d
order_status,str,0,0.0,8,delivered
order_purchase_timestamp,str,0,0.0,98875,2017-10-02 10:56:33
order_approved_at,str,160,0.2,90733,2017-10-02 11:07:15
order_delivered_carrier_date,str,1783,1.8,81018,2017-10-04 19:55:00
order_delivered_customer_date,str,2965,3.0,95664,2017-10-10 21:25:13
order_estimated_delivery_date,str,0,0.0,459,2017-10-18 00:00:00


... (8 outputs left out)


===== sellers  (3095, 4)  doublons complets : 0


,dtype,manquants,% manquants,valeurs distinctes,exemple
seller_id,str,0,0.0,3095,3442f8959a84dea7ee197c632cb2df15
seller_zip_code_prefix,str,0,0.0,2246,13023
seller_city,str,0,0.0,611,campinas
seller_state,str,0,0.0,23,SP



===== translation  (71, 2)  doublons complets : 0


,dtype,manquants,% manquants,valeurs distinctes,exemple
product_category_name,str,0,0.0,71,beleza_saude
product_category_name_english,str,0,0.0,71,health_beauty


## 3. Comment les tables sont liées entre elles ?

On cherche pour chaque table sa **clé** (colonne unique) et les **clés étrangères** vers les autres tables.

In [4]:
def est_cle(df, cols):
    return not df.duplicated(cols).any()

cles = {
    "customers.customer_id": est_cle(customers, ["customer_id"]),
    "customers.customer_unique_id": est_cle(customers, ["customer_unique_id"]),
    "orders.order_id": est_cle(orders, ["order_id"]),
    "orders.customer_id": est_cle(orders, ["customer_id"]),
    "order_items.(order_id, order_item_id)": est_cle(order_items, ["order_id", "order_item_id"]),
    "payments.(order_id, payment_sequential)": est_cle(payments, ["order_id", "payment_sequential"]),
    "reviews.review_id": est_cle(reviews, ["review_id"]),
    "reviews.(review_id, order_id)": est_cle(reviews, ["review_id", "order_id"]),
    "products.product_id": est_cle(products, ["product_id"]),
    "sellers.seller_id": est_cle(sellers, ["seller_id"]),
    "geolocation.zip_code_prefix": est_cle(geolocation, ["geolocation_zip_code_prefix"]),
    "translation.product_category_name": est_cle(translation, ["product_category_name"]),
}
pd.Series(cles, name="unique ?")

customers.customer_id                       True
customers.customer_unique_id               False
orders.order_id                             True
orders.customer_id                          True
order_items.(order_id, order_item_id)       True
payments.(order_id, payment_sequential)     True
reviews.review_id                          False
reviews.(review_id, order_id)               True
products.product_id                         True
sellers.seller_id                           True
geolocation.zip_code_prefix                False
translation.product_category_name           True
Name: unique ?, dtype: bool

In [5]:
def couverture(enfant, col_e, parent, col_p):
    """Part des valeurs de enfant[col_e] retrouvées dans parent[col_p]."""
    return round(enfant[col_e].isin(parent[col_p]).mean() * 100, 2)

liens = [
    ("orders.customer_id → customers", couverture(orders, "customer_id", customers, "customer_id")),
    ("order_items.order_id → orders", couverture(order_items, "order_id", orders, "order_id")),
    ("order_items.product_id → products", couverture(order_items, "product_id", products, "product_id")),
    ("order_items.seller_id → sellers", couverture(order_items, "seller_id", sellers, "seller_id")),
    ("payments.order_id → orders", couverture(payments, "order_id", orders, "order_id")),
    ("reviews.order_id → orders", couverture(reviews, "order_id", orders, "order_id")),
    ("products.category → translation", couverture(products.dropna(subset=["product_category_name"]), "product_category_name", translation, "product_category_name")),
    ("customers.zip → geolocation", couverture(customers, "customer_zip_code_prefix", geolocation, "geolocation_zip_code_prefix")),
    ("sellers.zip → geolocation", couverture(sellers, "seller_zip_code_prefix", geolocation, "geolocation_zip_code_prefix")),
    ("orders → order_items (commandes avec articles)", couverture(orders, "order_id", order_items, "order_id")),
    ("orders → payments (commandes payées)", couverture(orders, "order_id", payments, "order_id")),
    ("orders → reviews (commandes notées)", couverture(orders, "order_id", reviews, "order_id")),
]
pd.DataFrame(liens, columns=["lien", "% retrouvé"])

,lien,% retrouvé
0,orders.customer_id → customers,100.00
1,order_items.order_id → orders,100.00
2,order_items.product_id → products,100.00
3,order_items.seller_id → sellers,100.00
4,payments.order_id → orders,100.00
5,reviews.order_id → orders,100.00
6,products.category → translation,99.96
7,customers.zip → geolocation,99.72
8,sellers.zip → geolocation,99.77
9,orders → order_items (commandes avec articles),99.22


### Cardinalités

- 1 commande ↔ 1 `customer_id` : `customer_id` est un identifiant **par commande**, pas par personne.
- La personne réelle est `customer_unique_id` → c'est la clé du clustering.

In [6]:
par_personne = customers.groupby("customer_unique_id")["customer_id"].nunique()
print("customer_id :", customers.customer_id.nunique(), "| personnes (customer_unique_id) :", par_personne.size)
print(par_personne.value_counts().sort_index().head(10).to_string())

customer_id : 99441 | personnes (customer_unique_id) : 96096
customer_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1


In [7]:
cardinalites = pd.DataFrame({
    "max par commande": {
        "articles": order_items.groupby("order_id").size().max(),
        "paiements": payments.groupby("order_id").size().max(),
        "avis": reviews.groupby("order_id").size().max(),
        "vendeurs": order_items.groupby("order_id").seller_id.nunique().max(),
    },
    "commandes concernées (>1)": {
        "articles": (order_items.groupby("order_id").size() > 1).sum(),
        "paiements": (payments.groupby("order_id").size() > 1).sum(),
        "avis": (reviews.groupby("order_id").size() > 1).sum(),
        "vendeurs": (order_items.groupby("order_id").seller_id.nunique() > 1).sum(),
    },
})
cardinalites

,max par commande,commandes concernées (>1)
articles,21,9803
paiements,29,2961
avis,3,547
vendeurs,5,1278


In [8]:
# Commandes sans article : quel statut ?
orders.loc[~orders.order_id.isin(order_items.order_id), "order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [9]:
# Catégories sans traduction
sorted(set(products.product_category_name.dropna()) - set(translation.product_category_name))

['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']

## 4. Quelles informations sont importantes dans chaque colonne ?

### 4.1 orders : statuts et dates

In [10]:
orders.order_status.value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [11]:
dates = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
         "order_delivered_customer_date", "order_estimated_delivery_date"]
o = orders.copy()
o[dates] = o[dates].apply(pd.to_datetime)
print("période :", o.order_purchase_timestamp.min(), "→", o.order_purchase_timestamp.max())
o.order_purchase_timestamp.dt.to_period("M").value_counts().sort_index()

période : 2016-09-04 21:15:19 → 2018-10-17 17:30:18


order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

In [12]:
incoherences = {
    "validée avant achat": (o.order_approved_at < o.order_purchase_timestamp).sum(),
    "remise transporteur avant achat": (o.order_delivered_carrier_date < o.order_purchase_timestamp).sum(),
    "livrée avant remise transporteur": (o.order_delivered_customer_date < o.order_delivered_carrier_date).sum(),
    "statut delivered sans date de livraison": ((o.order_status == "delivered") & o.order_delivered_customer_date.isna()).sum(),
    "date de livraison mais statut ≠ delivered": ((o.order_status != "delivered") & o.order_delivered_customer_date.notna()).sum(),
}
pd.Series(incoherences, name="lignes")

validée avant achat                            0
remise transporteur avant achat              166
livrée avant remise transporteur              23
statut delivered sans date de livraison        8
date de livraison mais statut ≠ delivered      6
Name: lignes, dtype: int64

In [13]:
# Information dérivée importante : délai et retard de livraison
livrees = o.dropna(subset=["order_delivered_customer_date"])
delai = (livrees.order_delivered_customer_date - livrees.order_purchase_timestamp).dt.days
retard = livrees.order_delivered_customer_date > livrees.order_estimated_delivery_date
print(delai.describe().round(1).to_string())
print("part livrée en retard :", round(retard.mean() * 100, 1), "%")

count    96476.0
mean        12.1
std          9.6
min          0.0
25%          6.0
50%         10.0
75%         15.0
max        209.0
part livrée en retard : 8.1 %


### 4.2 order_items : prix et frais de port

In [14]:
order_items[["order_item_id", "price", "freight_value"]].describe().round(2)

,order_item_id,price,freight_value
count,112650.00,112650.00,112650.00
mean,1.20,120.65,19.99
std,0.71,183.63,15.81
min,1.00,0.85,0.00
25%,1.00,39.90,13.08
50%,1.00,74.99,16.26
75%,1.00,134.90,21.15
max,21.00,6735.00,409.68


In [15]:
print("frais de port à 0 :", (order_items.freight_value == 0).sum())
print("même produit répété dans une commande :", order_items.duplicated(["order_id", "product_id"]).sum())

frais de port à 0 : 383
même produit répété dans une commande : 10225


### 4.3 payments

In [16]:
display(payments.payment_type.value_counts())
display(payments[["payment_sequential", "payment_installments", "payment_value"]].describe().round(2))
print("paiements à 0 :", (payments.payment_value == 0).sum(), "| échéances à 0 :", (payments.payment_installments == 0).sum())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

,payment_sequential,payment_installments,payment_value
count,103886.00,103886.00,103886.00
mean,1.09,2.85,154.10
std,0.71,2.69,217.49
min,1.00,0.00,0.00
25%,1.00,1.00,56.79
50%,1.00,1.00,100.00
75%,1.00,4.00,171.84
max,29.00,24.00,13664.08


paiements à 0 : 9 | échéances à 0 : 2


In [17]:
# Cohérence entre fichiers : total articles + port == total payé ?
total_articles = order_items.groupby("order_id")[["price", "freight_value"]].sum().sum(axis=1)
total_paye = payments.groupby("order_id").payment_value.sum()
ecart = (total_articles - total_paye).dropna().abs()
print("commandes comparées :", len(ecart), "| écart > 1 R$ :", (ecart > 1).sum())

commandes comparées : 98665 | écart > 1 R$ : 249


### 4.4 reviews

In [18]:
display(reviews.review_score.value_counts().sort_index())
print("titre manquant :", round(reviews.review_comment_title.isna().mean() * 100, 1), "%")
print("message manquant :", round(reviews.review_comment_message.isna().mean() * 100, 1), "%")
print("review_id en double :", reviews.review_id.duplicated().sum())

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

titre manquant : 88.3 %
message manquant : 58.7 %
review_id en double : 814


In [19]:
# Un même review_id partagé par plusieurs commandes : même note ?
dup = reviews[reviews.review_id.duplicated(keep=False)]
dup.groupby("review_id").agg(nb_commandes=("order_id", "nunique"), notes=("review_score", "nunique")).describe()

,nb_commandes,notes
count,789.000000,789.0
mean,2.031686,1.0
std,0.175273,0.0
min,2.000000,1.0
25%,2.000000,1.0
50%,2.000000,1.0
75%,2.000000,1.0
max,3.000000,1.0


### 4.5 products

In [20]:
products.describe().round(1)

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.0,32341.0,32341.0,32949.0,32949.0,32949.0,32949.0
mean,48.5,771.5,2.2,2276.5,30.8,16.9,23.2
std,10.2,635.1,1.7,4282.0,16.9,13.6,12.1
min,5.0,4.0,1.0,0.0,7.0,2.0,6.0
25%,42.0,339.0,1.0,300.0,18.0,8.0,15.0
50%,51.0,595.0,1.0,700.0,25.0,13.0,20.0
75%,57.0,972.0,3.0,1900.0,38.0,21.0,30.0
max,76.0,3992.0,20.0,40425.0,105.0,105.0,118.0


In [21]:
print("poids = 0 g :", (products.product_weight_g == 0).sum())
print("catégorie manquante :", products.product_category_name.isna().sum())
# Les colonnes *_lenght (sic) décrivent la fiche produit, pas le produit lui-même.

poids = 0 g : 4
catégorie manquante : 610


### 4.6 customers / sellers : villes et États

In [22]:
print("États clients :", customers.customer_state.nunique(), "| États vendeurs :", sellers.seller_state.nunique())
villes_bizarres = sellers.seller_city[sellers.seller_city.str.contains(r"[^a-z ]", regex=True)]
villes_bizarres.unique()[:25]

États clients : 27 | États vendeurs : 23


<StringArray>
[                              'lages - sc',                             'auriflama/sp',                    'sao paulo / sao paulo',                               'são paulo',
                    'santa barbara d´oeste',                                 '04482255', 'novo hamburgo, rio grande do sul, brasil',                           'cariacica / es',
                       'sao miguel d'oeste',                           'sao paulo - sp',                                   'sbc/sp',           'arraial d'ajuda (porto seguro)',
                    'santo andre/sao paulo',                                  'sp / sp',                           'maua/sao paulo',                     'mogi das cruzes / sp',
           'rio de janeiro \rio de janeiro',                  'barbacena/ minas gerais',                                'andira-pr',          'rio de janeiro / rio de janeiro',
                               'pinhais/pr',                    'santa barbara d'oeste',             

### 4.7 geolocation

In [23]:
print("lignes :", len(geolocation), "| préfixes distincts :", geolocation.geolocation_zip_code_prefix.nunique())
print("doublons complets :", geolocation.duplicated().sum())
print("lignes par préfixe (moyenne) :", round(geolocation.groupby("geolocation_zip_code_prefix").size().mean(), 1))
hors_bresil = ~geolocation.geolocation_lat.between(-34, 6) | ~geolocation.geolocation_lng.between(-74, -34)
print("points hors du Brésil :", hors_bresil.sum())
# Orthographes multiples pour une même ville
geolocation.groupby("geolocation_zip_code_prefix").geolocation_city.nunique().sort_values(ascending=False).head()

lignes : 1000163 | préfixes distincts : 19015


doublons complets : 261831
lignes par préfixe (moyenne) : 52.6
points hors du Brésil : 42


geolocation_zip_code_prefix
13457    5
42850    5
17970    5
13318    5
78290    5
Name: geolocation_city, dtype: int64

## 5. Comment garder l'information importante ? Les agrégations

Les tables « filles » (articles, paiements, avis, géolocalisation) ont plusieurs lignes par clé.
On les **résume** avec `groupby().agg()` pour obtenir une ligne par commande (ou par code postal),
qu'on peut ensuite joindre sans dupliquer les lignes.

In [24]:
articles_par_commande = order_items.groupby("order_id").agg(
    nb_articles=("order_item_id", "count"),
    nb_produits_distincts=("product_id", "nunique"),
    nb_vendeurs=("seller_id", "nunique"),
    montant_articles=("price", "sum"),
    montant_port=("freight_value", "sum"),
)
articles_par_commande.head()

,nb_articles,nb_produits_distincts,nb_vendeurs,montant_articles,montant_port
order_id,,,,,
00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93
000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87
00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14


In [25]:
paiements_par_commande = payments.groupby("order_id").agg(
    montant_paye=("payment_value", "sum"),
    nb_paiements=("payment_sequential", "count"),
    nb_echeances_max=("payment_installments", "max"),
    # moyen de paiement principal = celui qui porte le plus gros montant
    moyen_principal=("payment_type", lambda s: s.loc[payments.loc[s.index, "payment_value"].idxmax()]),
)
paiements_par_commande.head()

,montant_paye,nb_paiements,nb_echeances_max,moyen_principal
order_id,,,,
00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2,credit_card
00018f77f2f0320c557190d7a144bdd3,259.83,1,3,credit_card
000229ec398224ef6ca0657da4fc703e,216.87,1,5,credit_card
00024acbcdf0a6daa1e931b038114c75,25.78,1,2,credit_card
00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3,credit_card


In [26]:
avis_par_commande = reviews.groupby("order_id").agg(
    note_moyenne=("review_score", "mean"),
    nb_avis=("review_id", "nunique"),
    a_commente=("review_comment_message", lambda s: s.notna().any()),
)
avis_par_commande.head()

,note_moyenne,nb_avis,a_commente
order_id,,,
00010242fe8c5a6d1ba2dd792cb16214,5.0,1,True
00018f77f2f0320c557190d7a144bdd3,4.0,1,False
000229ec398224ef6ca0657da4fc703e,5.0,1,True
00024acbcdf0a6daa1e931b038114c75,4.0,1,False
00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1,True


In [27]:
# Géolocalisation : 1 point par préfixe (médiane, robuste aux points aberrants)
geo_par_cp = (geolocation[~hors_bresil]
              .groupby("geolocation_zip_code_prefix")
              .agg(lat=("geolocation_lat", "median"), lng=("geolocation_lng", "median"), nb_points=("geolocation_lat", "size")))
print(len(geolocation), "→", len(geo_par_cp), "lignes")
geo_par_cp.head()

1000163 → 19010 lignes


,lat,lng,nb_points
geolocation_zip_code_prefix,,,
01001,-23.550381,-46.634027,26
01002,-23.548551,-46.635072,13
01003,-23.548977,-46.635313,17
01004,-23.549535,-46.634771,22
01005,-23.549612,-46.636532,25


In [28]:
# Vue « une ligne par commande » : la jointure ne doit pas multiplier les lignes
commandes = (orders
             .merge(customers, on="customer_id", how="left", validate="one_to_one")
             .merge(articles_par_commande, on="order_id", how="left", validate="one_to_one")
             .merge(paiements_par_commande, on="order_id", how="left", validate="one_to_one")
             .merge(avis_par_commande, on="order_id", how="left", validate="one_to_one"))
assert len(commandes) == len(orders)
commandes.shape

(99441, 24)

In [29]:
# Aperçu de l'agrégation finale à l'échelle de la personne (préfigure le RFM de l'étape 6)
c = commandes[commandes.order_status == "delivered"].copy()
c["order_purchase_timestamp"] = pd.to_datetime(c.order_purchase_timestamp)
reference = c.order_purchase_timestamp.max()
par_client = c.groupby("customer_unique_id").agg(
    recence_jours=("order_purchase_timestamp", lambda s: (reference - s.max()).days),
    frequence=("order_id", "nunique"),
    montant_total=("montant_paye", "sum"),
    note_moyenne=("note_moyenne", "mean"),
)
par_client["panier_moyen"] = par_client.montant_total / par_client.frequence
par_client.describe().round(1)

,recence_jours,frequence,montant_total,note_moyenne,panier_moyen
count,93358.0,93358.0,93358.0,92755.0,93358.0
mean,236.9,1.0,165.2,4.2,160.3
std,152.6,0.2,226.3,1.3,219.6
min,0.0,1.0,0.0,1.0,0.0
25%,113.0,1.0,63.1,4.0,62.4
50%,218.0,1.0,107.8,5.0,105.6
75%,345.0,1.0,182.6,5.0,176.6
max,713.0,15.0,13664.1,5.0,13664.1
